In [1]:
import pandas as pd
import numpy as np

print("pandas:", pd.__version__)
print("NumPy:", np.__version__)


pandas: 3.0.1
NumPy: 2.3.5


In [2]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df_original = pd.read_csv(url)
print(df_original.shape)
print(df_original.head())


(10000, 11)
   model_year  origin  ... acceleration fuel_efficiency_mpg
0        2006  Europe  ...          NaN                31.9
1        2008  Europe  ...          NaN                31.3
2        1996    Asia  ...         17.3                27.5
3        1989  Europe  ...         18.7                28.5
4        1994     USA  ...         17.5                31.0

[5 rows x 11 columns]


In [3]:
features = [
    "engine_displacement",
    "horsepower",
    "vehicle_weight",
    "model_year",
]
target = "fuel_efficiency_mpg"
columns = features + [target]
df = df_original[columns].copy()
print(df.shape)


(10000, 5)


In [4]:
print(df[target].describe())
print("Assimetria:", df[target].skew())

counts, edges = np.histogram(df[target].to_numpy(), bins=20)
print("\nHistograma: intervalo em mpg | quantidade | barra (bloco ≈ 50 carros)")
for count, start, end in zip(counts, edges[:-1], edges[1:]):
    bar = "█" * int(count / 50)
    print(f"{start:5.2f} a {end:5.2f} | {count:4d} | {bar}")


count    10000.000000
mean        29.997700
std          2.930245
min         19.800000
25%         28.000000
50%         30.000000
75%         31.900000
max         41.200000
Name: fuel_efficiency_mpg, dtype: float64
Assimetria: 0.08080419601878173

Histograma: intervalo em mpg | quantidade | barra (bloco ≈ 50 carros)
19.80 a 20.87 |    4 | 
20.87 a 21.94 |   15 | 
21.94 a 23.01 |   58 | █
23.01 a 24.08 |   99 | █
24.08 a 25.15 |  289 | █████
25.15 a 26.22 |  528 | ██████████
26.22 a 27.29 |  758 | ███████████████
27.29 a 28.36 | 1192 | ███████████████████████
28.36 a 29.43 | 1362 | ███████████████████████████
29.43 a 30.50 | 1310 | ██████████████████████████
30.50 a 31.57 | 1454 | █████████████████████████████
31.57 a 32.64 | 1151 | ███████████████████████
32.64 a 33.71 |  741 | ██████████████
33.71 a 34.78 |  446 | ████████
34.78 a 35.85 |  324 | ██████
35.85 a 36.92 |  175 | ███
36.92 a 37.99 |   58 | █
37.99 a 39.06 |   28 | 
39.06 a 40.13 |    7 | 
40.13 a 41.20 |    1 | 


In [5]:
missing = df.isna().sum()
print(missing)
q1_answer = missing[missing > 0].index[0]
print("Resposta Q1:", q1_answer)


engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64
Resposta Q1: horsepower


In [6]:
q2_answer = df["horsepower"].median()
print("Resposta Q2:", q2_answer)


Resposta Q2: 254.0


In [7]:
def split_data(data, seed):
    n = len(data)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    train = data.iloc[idx[:n_train]].copy()
    val = data.iloc[idx[n_train:n_train + n_val]].copy()
    test = data.iloc[idx[n_train + n_val:]].copy()
    return train, val, test


In [8]:
df_train, df_val, df_test = split_data(df, seed=42)
y_train = df_train[target].to_numpy()
y_val = df_val[target].to_numpy()
print("Treino:", len(df_train), "Validação:", len(df_val), "Teste:", len(df_test))


Treino: 6000 Validação: 2000 Teste: 2000


In [9]:
def prepare_X(data, fill_value=0):
    selected = data[features].copy()
    filled = selected.fillna(fill_value)
    X = filled.to_numpy(dtype=float)
    return X


In [10]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]


In [11]:
def rmse(y, y_pred):
    errors = y - y_pred
    squared_errors = errors ** 2
    mse = squared_errors.mean()
    return np.sqrt(mse)


In [12]:
mean_horsepower = df_train["horsepower"].mean()
print("Média da potência no treino:", mean_horsepower)
q3_rows = []

for method, fill_value in [("zero", 0), ("média", mean_horsepower)]:
    X_train = prepare_X(df_train, fill_value)
    X_val = prepare_X(df_val, fill_value)
    w0, w = train_linear_regression(X_train, y_train)
    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)
    q3_rows.append({"preenchimento": method, "rmse": score, "rmse_3": round(float(score), 3)})

q3_results = pd.DataFrame(q3_rows)
print(q3_results.to_string(index=False))
zero_score, mean_score = q3_results["rmse_3"].to_numpy()
if zero_score < mean_score:
    q3_answer = "With 0"
elif mean_score < zero_score:
    q3_answer = "With mean"
else:
    q3_answer = "Both are equally good"
print("Resposta Q3:", q3_answer)


Média da potência no treino: 254.46338337605272
preenchimento     rmse  rmse_3
         zero 2.205293   2.205
        média 2.201817   2.202
Resposta Q3: With mean


In [13]:
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    identity = np.eye(XTX.shape[0])
    XTX = XTX + r * identity
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]


In [14]:
X_train_zero = prepare_X(df_train, fill_value=0)
X_val_zero = prepare_X(df_val, fill_value=0)
r_values = [0, 0.01, 0.1, 1, 5, 10, 100]
q4_rows = []

for r in r_values:
    w0, w = train_linear_regression_reg(X_train_zero, y_train, r=r)
    y_pred = w0 + X_val_zero.dot(w)
    score = rmse(y_val, y_pred)
    q4_rows.append({"r": r, "rmse": score, "rmse_4": round(float(score), 4)})

q4_results = pd.DataFrame(q4_rows)
print(q4_results.to_string(index=False))
best_q4 = q4_results.sort_values(["rmse_4", "r"]).iloc[0]
q4_answer = float(best_q4["r"])
print("Resposta Q4: r =", q4_answer)


     r     rmse  rmse_4
  0.00 2.205293  2.2053
  0.01 2.205828  2.2058
  0.10 2.224143  2.2241
  1.00 2.349229  2.3492
  5.00 2.409380  2.4094
 10.00 2.419470  2.4195
100.00 2.429202  2.4292
Resposta Q4: r = 0.0


In [15]:
q5_rows = []
scores = []

for seed in range(10):
    train_seed, val_seed, test_seed = split_data(df, seed=seed)
    y_train_seed = train_seed[target].to_numpy()
    y_val_seed = val_seed[target].to_numpy()
    X_train_seed = prepare_X(train_seed, fill_value=0)
    X_val_seed = prepare_X(val_seed, fill_value=0)
    w0, w = train_linear_regression(X_train_seed, y_train_seed)
    y_pred_seed = w0 + X_val_seed.dot(w)
    score = rmse(y_val_seed, y_pred_seed)
    scores.append(float(score))
    q5_rows.append({"seed": seed, "rmse": score})

print(pd.DataFrame(q5_rows).to_string(index=False))
std_scores = np.std(scores)
q5_answer = round(float(std_scores), 3)
print("Desvio padrão sem arredondar:", std_scores)
print("Resposta Q5:", q5_answer)


 seed     rmse
    0 2.239204
    1 2.202113
    2 2.163420
    3 2.204359
    4 2.201880
    5 2.245785
    6 2.269721
    7 2.190795
    8 2.216611
    9 2.207037
Desvio padrão sem arredondar: 0.028781274078191175
Resposta Q5: 0.029


In [16]:
train_9, val_9, test_9 = split_data(df, seed=9)
df_full_train = pd.concat([train_9, val_9], ignore_index=True)
y_full_train = df_full_train[target].to_numpy()
y_test = test_9[target].to_numpy()
X_full_train = prepare_X(df_full_train, fill_value=0)
X_test = prepare_X(test_9, fill_value=0)
w0_final, w_final = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)
y_pred_test = w0_final + X_test.dot(w_final)
q6_rmse = rmse(y_test, y_pred_test)
q6_answer = round(float(q6_rmse), 3)

print("X de treino final:", X_full_train.shape, "X de teste:", X_test.shape)
print("Intercepto aprendido:", w0_final)
print("Pesos por feature:\n", pd.Series(w_final, index=features))
print("RMSE de teste:", q6_rmse)
print("Resposta Q6:", q6_answer)


X de treino final: (8000, 4) X de teste: (2000, 4)
Intercepto aprendido: -152.37535772493302
Pesos por feature:
 engine_displacement   -0.001355
horsepower            -0.000426
vehicle_weight        -0.004018
model_year             0.101404
dtype: float64
RMSE de teste: 2.235827747191731
Resposta Q6: 2.236


In [17]:
answers = {
    "Q1": q1_answer,
    "Q2": q2_answer,
    "Q3": q3_answer,
    "Q4": q4_answer,
    "Q5": q5_answer,
    "Q6": q6_answer,
}
for question, answer in answers.items():
    print(f"{question}: {answer}")


Q1: horsepower
Q2: 254.0
Q3: With mean
Q4: 0.0
Q5: 0.029
Q6: 2.236
